In [1]:
from catboost import CatBoostClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import numpy as np
import json
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [31]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
seed = 42
np.random.seed(seed)

In [4]:
import re


def extract_text(msg):
    if not isinstance(msg, list):
        return msg

    result_text = ''
    for item in msg:
        if isinstance(item, str):
            result_text += (item + " ")
        else:
            result_text += (item['text'] + " ")
    return result_text

def prepoccess_dataset(df):
    preproccessed_df = df.copy()
    preproccessed_df = df.dropna(subset=['text'])
    preproccessed_df.reset_index(drop=True, inplace=True)
    preproccessed_df['text'] = preproccessed_df['text'].map(lambda x: extract_text(x))
    preproccessed_df['text_clean_nums'] = preproccessed_df['text'].map(lambda x: re.sub(r'\d', '', x))
    preproccessed_df = preproccessed_df[preproccessed_df['text_clean_nums'] != '']
    return preproccessed_df

def chat_to_df(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data_json = json.load(f)

    user_id = []
    usernames = []
    dates = []
    texts = []
    for msg in data_json['messages']:
        if msg['type'] == 'message':
            user_id.append(msg['from_id'])
            if 'forwarded_from' in msg.keys():
                usernames.append(msg['forwarded_from'])
            else:
                usernames.append(msg['from'])
            dates.append(msg['date'])
            texts.append(msg['text'])

    df = pd.DataFrame({
        'user_id': user_id,
        'username': usernames,
        'date': dates,
        'text': texts,
    })
    
    return prepoccess_dataset(df)

# Загрузка датафрейма

In [5]:
json_path = r'../../data/chats/marakasi_14.02.2026.json'
df = chat_to_df(json_path)

In [6]:
df.head()

,user_id,username,date,text,text_clean_nums
0,user674227701,свитистер МТС,2024-08-21T22:53:00,А мне,А мне
1,user790422015,Ири,2024-08-21T22:53:02,💋💋💋,💋💋💋
2,user790422015,Ири,2024-08-21T22:53:15,Пожалуйста не надо альфа Лера,Пожалуйста не надо альфа Лера
3,user872595910,ВАЛЬРА!,2024-08-21T22:53:20,Да,Да
4,user790422015,Ири,2024-08-21T22:53:23,А что ты хочешь,А что ты хочешь


In [7]:
df = df.drop(columns=['user_id', 'date'])

In [8]:
marakasi_users = [
    'свитистер МТС', 'Ири', 'ВАЛЬРА!', 'Шорт', 'Арина Колошек🌏',
    'Соня🤘', 'Егорчек🙄', 'Denis ss', 'dandelions', 'Умка', 'nuta',
    '見知らぬ人 アイ', 'Кибержаба'
]

marakasi_users = [
    'свитистер МТС', 'Ири', 'ВАЛЬРА!', 'Шорт', 'Арина Колошек🌏',
    'Соня🤘', 'Егорчек🙄', 'Denis ss', 'dandelions', 'Умка', 'nuta',
]

In [9]:
df = df[df['username'].isin(marakasi_users)]

In [10]:
df['username'].value_counts()

username
dandelions        16662
свитистер МТС     10964
ВАЛЬРА!           10634
Соня🤘              4487
Шорт               4269
Егорчек🙄           3695
Ири                2461
Denis ss           2387
Арина Колошек🌏     2268
Умка               2161
nuta               2023
Name: count, dtype: int64

In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['username'])

In [13]:
id2label = {int(i): str(label) for i, label in enumerate(le.classes_)}
label2id = {str(label): int(i) for i, label in enumerate(le.classes_)}

In [14]:
id2label

{0: 'Denis ss',
 1: 'dandelions',
 2: 'nuta',
 3: 'Арина Колошек🌏',
 4: 'ВАЛЬРА!',
 5: 'Егорчек🙄',
 6: 'Ири',
 7: 'Соня🤘',
 8: 'Умка',
 9: 'Шорт',
 10: 'свитистер МТС'}

In [15]:
df = df.drop(columns=['text', 'username'])

In [16]:
df.head()

,text_clean_nums,label
0,А мне,10
1,💋💋💋,6
2,Пожалуйста не надо альфа Лера,6
3,Да,4
4,А что ты хочешь,6


In [18]:
model_id = 'DeepPavlov/rubert-base-cased-conversational'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(df['label'].unique()),
    id2label=id2label, label2id=label2id,
    device_map='auto', dtype='auto'
)

W0329 20:57:21.288000 8732 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased-conversational and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
def get_embedding_from_df(df, column, model, tokenizer, batch_size=32):
    model.eval()

    all_embeddings = []

    for i in tqdm(range(0, df.shape[0], batch_size)):
        batch_texts = df[column][i : i + batch_size].tolist()
        
        inputs = tokenizer(batch_texts, return_tensors='pt', padding=True, max_length=128, truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model.base_model(**inputs)
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(embeddings)
    return np.vstack(all_embeddings)

In [20]:
text_embeddings = get_embedding_from_df(df, 'text_clean_nums', model, tokenizer)

100%|██████████████████████████████████████████████████████████████████████████████| 1938/1938 [01:11<00:00, 26.93it/s]


In [21]:
df['embedding'] = list(text_embeddings)

In [22]:
X = df.drop(columns=['label'])
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [40]:
cb = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,          
    depth=6,                     
    l2_leaf_reg=3, 
    loss_function='MultiClass',
    custom_metric=['Precision', 'Recall', 'F1', 'Accuracy'],
    
    # Работа с признаками
    embedding_features=['embedding'],
    text_features=['text_clean_nums'],
    
    # Автоматика
    early_stopping_rounds=50,     
    use_best_model=True,          
    
    # Ускорение и логи               
    task_type="GPU",              # ОБЯЗАТЕЛЬНО, если есть видеокарта
    devices='0',                  # Индекс GPU
    verbose=100
)

In [42]:
id2label

{0: 'Denis ss',
 1: 'dandelions',
 2: 'nuta',
 3: 'Арина Колошек🌏',
 4: 'ВАЛЬРА!',
 5: 'Егорчек🙄',
 6: 'Ири',
 7: 'Соня🤘',
 8: 'Умка',
 9: 'Шорт',
 10: 'свитистер МТС'}

In [41]:
cb.fit(X_train, y_train, eval_set=(X_test, y_test), plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Set

0:	learn: 2.3387603	test: 2.3367934	best: 2.3367934 (0)	total: 376ms	remaining: 6m 15s


Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Set

100:	learn: 1.6238647	test: 1.5945896	best: 1.5945896 (100)	total: 5.27s	remaining: 46.9s


...

200:	learn: 1.5769150	test: 1.5486798	best: 1.5486798 (200)	total: 9.88s	remaining: 39.3s


Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Set

300:	learn: 1.5539999	test: 1.5316037	best: 1.5316037 (300)	total: 14.5s	remaining: 33.7s


Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Set

400:	learn: 1.5393924	test: 1.5229476	best: 1.5229476 (400)	total: 19.1s	remaining: 28.5s
500:	learn: 1.5286728	test: 1.5167163	best: 1.5167163 (500)	total: 23.6s	remaining: 23.5s
600:	learn: 1.5203217	test: 1.5132195	best: 1.5132195 (600)	total: 28s	remaining: 18.6s
700:	learn: 1.5141382	test: 1.5105284	best: 1.5105284 (700)	total: 32.4s	remaining: 13.8s
800:	learn: 1.5086446	test: 1.5087054	best: 1.5087054 (800)	total: 36.7s	remaining: 9.12s
900:	learn: 1.5035858	test: 1.5072843	best: 1.5072843 (900)	total: 41.3s	remaining: 4.53s
999:	learn: 1.4986593	test: 1.5060548	best: 1.5060548 (999)	total: 45.8s	remaining: 0us
bestTest = 1.506054798
bestIteration = 999
